# Generation Expansion DBC/iDBC Notebook

Use this notebook to run minimal generation-expansion experiments with either `DBC` or `iDBC`.


In [1]:
using Pkg

function find_repo_root(start::AbstractString = pwd())
    dir = abspath(start)
    while true
        has_project = isfile(joinpath(dir, "Project.toml"))
        has_entry = isfile(joinpath(dir, "experiments", "generation_expansion", "run_experiment.jl"))
        if has_project && has_entry
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repository root from $(start).")
        dir = parent
    end
end

repo_root = find_repo_root()
cd(repo_root)
Pkg.activate(repo_root)
println("repo_root = ", repo_root)


  Activating 

repo_root = /Users/aaron/StoCutDyProg

project at `~/StoCutDyProg`


## Edit Parameters

`cut_mode` can be `:DBC` or `:iDBC`.


In [2]:
run_script = joinpath("experiments", "generation_expansion", "run_experiment.jl")
config_path = joinpath("notebooks", "tmp_ge_notebook_config.jl")

cut_mode = :iDBC                     # :DBC or :iDBC
enable_copy_branching = (cut_mode == :iDBC)
enable_surrogate_copy_split = true
idbc_warm_pass = false

periods = 6
realizations = 6

num_workers = 5
dry_run = false
max_iter = 5
sample_count = 20
num_backward_samples = 1
terminate_time = 300.0
time_limit = 20.0
experiment_tag = "nb_ge_$(cut_mode)"

config_text = """
EXPERIMENT_CONFIG = GeExperimentConfig(
    num_workers = $(num_workers),
    dry_run = $(dry_run),
    static = GeStaticConfig(
        max_iter = $(max_iter),
        sample_count = $(sample_count),
        num_backward_samples = $(num_backward_samples),
        terminate_time = $(terminate_time),
        time_limit = $(time_limit),
        enable_copy_branching = $(enable_copy_branching),
        enable_surrogate_copy_split = $(enable_surrogate_copy_split),
        copy_split_strategy = :surrogate_delta,
        idbc_warm_pass = $(idbc_warm_pass),
        experiment_tag = $(repr(experiment_tag)),
        legacy_logger_paths = false,
    ),
    sweep = GeSweepConfig(
        cuts = Symbol[$(repr(cut_mode))],
        realizations = Int[$(realizations)],
        periods = Int[$(periods)],
    ),
)
"""

write(joinpath(repo_root, config_path), config_text)
println("Wrote config: ", joinpath(repo_root, config_path))
println(config_text)


Wrote config: /Users/aaron/StoCutDyProg/notebooks/tmp_ge_notebook_config.jl
EXPERIMENT_CONFIG = GeExperimentConfig(
    num_workers = 5,
    dry_run = false,
    static = GeStaticConfig(
        max_iter = 5,
        sample_count = 20,
        num_backward_samples = 1,
        terminate_time = 300.0,
        time_limit = 20.0,
        enable_copy_branching = true,
        enable_surrogate_copy_split = true,
        copy_split_strategy = :surrogate_delta,
        idbc_warm_pass = false,
        experiment_tag = "nb_ge_iDBC",
        legacy_logger_paths = false,
    ),
    sweep = GeSweepConfig(
        cuts = Symbol[:iDBC],
        realizations = Int[6],
        periods = Int[6],
    ),
)



In [3]:
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end


  Activating project at `~/StoCutDyProg`
ERROR: LoadError: MethodError: no method matching GeStaticConfig(; max_iter::Int64, sample_count::Int64, num_backward_samples::Int64, terminate_time::Float64, time_limit::Float64, enable_copy_branching::Bool, enable_surrogate_copy_split::Bool, copy_split_strategy::Symbol, idbc_warm_pass::Bool, experiment_tag::String, legacy_logger_paths::Bool)
This error has been manually thrown, explicitly, so the method may exist but be intentionally marked as unimplemented.

Closest candidates are:
  GeStaticConfig(; enhancement, enforce_binary_copies, sample_count, num_backward_samples, count_benders_cut, opt, logger_save, feasibility_tol, mip_focus, numeric_focus, mip_gap, disjunction_iteration_limit, branch_selection_strategy, copy_branch_policy, copy_branch_min_deviation, copy_branch_dominance_ratio, copy_branch_mean_ratio, copy_branch_boost, enable_copy_branching, enable_surrogate_copy_split, copy_split_strategy, copy_split_delta_tol, idbc_warm_pass, bra

ProcessFailedException: failed process: Process(`julia --project=. experiments/generation_expansion/run_experiment.jl notebooks/tmp_ge_notebook_config.jl`, ProcessExited(1)) [1]
